# 02 · How small can the panel get?
All numbers come from 5×5 repeated, patient-grouped CV, with gene selection redone inside every training fold.

In [ ]:
import os, sys
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
from genepanel.cli import load_config, load_datasets
cfg = load_config('config/default.yaml')
TAB = f"{cfg['results_dir']}/tables"; FIG = f"{cfg['results_dir']}/figures"
pd.set_option('display.precision', 4)

In [ ]:
summary = pd.read_csv(f'{TAB}/summary.csv')
ret = pd.read_csv(f'{TAB}/retention.csv')
vsr = pd.read_csv(f'{TAB}/vs_random.csv')

## AUC vs panel size
Dashed line: all genes. Gray band: 5–95% range of random gene panels of the same size (logistic regression).

In [ ]:
import glob
for f in sorted(glob.glob(f'{FIG}/auc_vs_k_*.png')):
    display(Image(f))

## Smallest panel that retains performance
Non-inferiority: the lower 95% bound of the paired fold-wise AUC difference (panel − all genes) must stay above −0.01.

In [ ]:
ret.pivot_table(index=['cancer', 'task', 'selector'], columns='model', values='smallest_retained_k')

## Do selected panels beat random genes?

In [ ]:
vsr[vsr.k.isin([1, 2, 5, 10])].pivot_table(index=['cancer', 'task', 'k'], columns='selector', values='auc')
  .join(vsr.groupby(['cancer', 'task', 'k']).random_auc_mean.first().rename('RANDOM'))

## Beyond AUC: sensitivity at 95% specificity
The clinically relevant operating point for screening.

In [ ]:
summary[summary.k.isin([1, 5, 10, 50]) & (summary.model == 'logreg') & (summary.selector != 'random')]\
  .pivot_table(index=['cancer', 'task', 'k'], columns='selector', values='sens_at_95spec_mean')

## Selector × model at k = 10

In [ ]:
display(Image(f'{FIG}/heatmap_k10.png'))

## Stability

In [ ]:
display(Image(f'{FIG}/stability.png'))

## Cross-cancer transfer of the consensus 10-gene panel

In [ ]:
pd.read_csv(f'{TAB}/transfer.csv')